# CNN Autoencoder Hyperparamter Optimzation for Sleep-EDF Hypnograms


In [1]:
from __future__ import annotations

import argparse
import json
import math
import os
import random
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import mne
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import umap

## Fixed project choices

In [ ]:
SEQ_LEN = 960                         # 8 h with 30 s epochs
EPOCH_SEC = 30.0
NUM_CLASSES = 6                       # W, N1, N2, N3, REM, Padding
PAD_CLASS = 5
SLEEP_STAGE_CLASSES = [0, 1, 2, 3, 4]
CLASS_TO_STAGE = {0: "W", 1: "N1", 2: "N2", 3: "N3", 4: "REM", 5: "PAD"}
STAGE_TO_CLASS = {v: k for k, v in CLASS_TO_STAGE.items()}
DEFAULT_CHANNELS = 32
DEFAULT_KERNEL_SIZE = 3
WEIGHT_DECAY = 1e-4

LATENT_DIM_CHOICES = [16, 32, 64, 128]
N_CONV_BLOCKS_CHOICES = [1, 2, 3]
FILTERS_BASE_CHOICES = [16, 32, 64]
KERNEL_SIZE_CHOICES = [3, 7, 11]
N_CLUSTERS_CHOICES = [2, 3, 4, 5, 6]
UMAP_N_NEIGHBORS_CHOICES = [5, 10, 15, 30]
UMAP_MIN_DIST_CHOICES = [0.0, 0.05, 0.1, 0.3]
UMAP_N_COMPONENTS_CHOICES = [2, 3, 5]
LEARNING_RATE_LOW = 1e-4
LEARNING_RATE_HIGH = 3e-3

# HPO objective uses clustering validity metrics >> Silhouette: [-1, 1] and Davies-Bouldin: transformed to 1 / (1 + DBI).
OBJECTIVE_SILHOUETTE_WEIGHT = 0.70
OBJECTIVE_DBI_WEIGHT = 0.30

VALID_CLUSTER_METHODS = ["kmeans", "gmm", "agglomerative"]


@dataclass
class NightRecord:
    record_id: str
    subject_id: str
    cohort: str
    path: str
    stages: List[int]  # unpadded integer stage sequence, values 0..4

## Reproducibility and utilities

In [ ]:
def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def infer_cohort(path: Path) -> str:
    name = path.name.upper()
    if name.startswith("SC"):
        return "SC"
    if name.startswith("ST"):
        return "ST"
    return "UNKNOWN"


def infer_subject_id(record_id: str) -> str:
    """Sleep-EDF convention: first 5 chars group repeated nights of one subject.

    Examples:
        SC4001E0 -> SC400
        SC4002E0 -> SC400
        ST7011J0 -> ST701
    """
    rid = record_id.upper()
    if len(rid) >= 5 and (rid.startswith("SC") or rid.startswith("ST")):
        return rid[:5]
    return rid


def normalize_stage_description(desc: str) -> Optional[int]:
    """Map MNE annotation descriptions to integer stage classes 0..4.

    Returns None for movement, unknown, unscored or non-stage annotations.
    """
    d = str(desc).strip().lower()
    d = d.replace("sleep stage", "").replace("stage", "").strip()
    d = d.replace("sleep_stage", "").strip()

    if d in {"w", "wake", "0"}:
        return STAGE_TO_CLASS["W"]
    if d in {"1", "n1"}:
        return STAGE_TO_CLASS["N1"]
    if d in {"2", "n2"}:
        return STAGE_TO_CLASS["N2"]
    if d in {"3", "n3", "n4"}:
        return STAGE_TO_CLASS["N3"]
    if d in {"4", "r", "rem"}:
        return STAGE_TO_CLASS["REM"]
    if d in {"m", "movement", "?", "unknown", "unscored", ""}:
        return None
    return None


def discover_fif_files(data_dir: Path) -> List[Path]:
    """Find preprocessed FIF files under the Sleep-EDF folder."""
    patterns = ["**/preprocessed/*-raw.fif", "**/preprocessed/*.fif"]
    files: List[Path] = []
    seen = set()
    for pattern in patterns:
        for f in sorted(data_dir.glob(pattern)):
            key = str(f.resolve())
            if key not in seen:
                files.append(f)
                seen.add(key)
    return files


def load_fif_hypnogram(path: Path) -> List[int]:
    raw = mne.io.read_raw_fif(str(path), preload=False, verbose=False)
    stages: List[int] = []

    for duration, desc in zip(raw.annotations.duration, raw.annotations.description):
        stage = normalize_stage_description(desc)
        if stage is None:
            continue
        n_epochs = int(round(float(duration) / EPOCH_SEC))
        if n_epochs > 0:
            stages.extend([stage] * n_epochs)

    return stages


def load_records(data_dir: Path, min_valid_epochs: int = 120) -> List[NightRecord]:
    fif_files = discover_fif_files(data_dir)
    if not fif_files:
        raise FileNotFoundError(
            f"No preprocessed FIF files found under {data_dir}. Expected files like '**/preprocessed/*-raw.fif'."
        )

    records: List[NightRecord] = []
    for path in fif_files:
        try:
            stages = load_fif_hypnogram(path)
        except Exception as exc:
            print(f"[WARN] Could not read {path}: {exc}")
            continue

        if len(stages) < min_valid_epochs:
            print(f"[WARN] Skipping {path.name}: only {len(stages)} valid epochs")
            continue

        record_id = re.sub(r"-raw\.fif$", "", path.name, flags=re.IGNORECASE)
        cohort = infer_cohort(path)
        subject_id = infer_subject_id(record_id)
        records.append(NightRecord(record_id, subject_id, cohort, str(path), stages))

    if not records:
        raise RuntimeError("No valid FIF records were loaded.")
    return records


def fixed_length_sequence(stages: List[int], seq_len: int = SEQ_LEN) -> np.ndarray:
    """Crop or pad to a fixed length. Padding is represented by PAD_CLASS=5."""
    arr = np.full(seq_len, PAD_CLASS, dtype=np.int64)
    n = min(len(stages), seq_len)
    arr[:n] = np.asarray(stages[:n], dtype=np.int64)
    return arr


def records_to_array(records: List[NightRecord], seq_len: int = SEQ_LEN) -> np.ndarray:
    return np.stack([fixed_length_sequence(r.stages, seq_len=seq_len) for r in records], axis=0)

## Dataset and model

In [4]:
class HypnogramDataset(torch.utils.data.Dataset):
    def __init__(self, X: np.ndarray):
        self.X = torch.as_tensor(X, dtype=torch.long)

    def __len__(self) -> int:
        return int(self.X.shape[0])

    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.X[idx]


class HypnoCNNAutoencoder(nn.Module):
    """CNN autoencoder for fixed-length hypnogram token sequences.

    The model receives integer class sequences [B, L], internally converts them
    to one-hot [B, C, L], compresses them to a latent vector z and reconstructs
    logits [B, C, L]. Padding is ignored by the loss.
    """

    def __init__(
        self,
        seq_len: int = SEQ_LEN,
        num_classes: int = NUM_CLASSES,
        latent_dim: int = 16,
        n_conv_blocks: int = 2,
        channels: int = DEFAULT_CHANNELS,
        kernel_size: int = DEFAULT_KERNEL_SIZE,
    ):
        super().__init__()
        self.seq_len = seq_len
        self.num_classes = num_classes
        self.latent_dim = latent_dim
        self.n_conv_blocks = n_conv_blocks
        self.channels = channels
        self.kernel_size = kernel_size

        pad = kernel_size // 2
        encoder_layers: List[nn.Module] = [
            nn.Conv1d(num_classes, channels, kernel_size=kernel_size, stride=1, padding=pad),
            nn.ReLU(),
            nn.Dropout(0.5),
        ]

        self.down_channels: List[int] = []
        in_ch = channels
        for b in range(n_conv_blocks):
            out_ch = channels if b == 0 else channels * (2 ** b)
            self.down_channels.append(out_ch)
            dropout = 0.5 if b == 0 else 0.3
            encoder_layers.extend([
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, stride=2, padding=pad),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            in_ch = out_ch

        self.encoder_conv = nn.Sequential(*encoder_layers)

        with torch.no_grad():
            dummy = torch.zeros(1, num_classes, seq_len)
            conv_out = self.encoder_conv(dummy)
        self.encoded_channels = int(conv_out.shape[1])
        self.encoded_len = int(conv_out.shape[2])
        self.flatten_dim = self.encoded_channels * self.encoded_len

        self.fc_latent = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, self.flatten_dim)

        decoder_layers: List[nn.Module] = []
        current_ch = self.encoded_channels
        # Reverse the downsampling path. Last transposed conv outputs NUM_CLASSES.
        for b in reversed(range(n_conv_blocks)):
            out_ch = num_classes if b == 0 else self.down_channels[b - 1]
            decoder_layers.append(
                nn.ConvTranspose1d(current_ch, out_ch, kernel_size=4, stride=2, padding=1)
            )
            if b != 0:
                decoder_layers.extend([nn.ReLU(), nn.Dropout(0.3)])
            current_ch = out_ch

        self.decoder_conv = nn.Sequential(*decoder_layers)

    def encode(self, x_tokens: torch.Tensor) -> torch.Tensor:
        x = F.one_hot(x_tokens.clamp(min=0, max=PAD_CLASS), num_classes=self.num_classes).float()
        x = x.permute(0, 2, 1)  # [B, C, L]
        h = self.encoder_conv(x)
        h = h.flatten(start_dim=1)
        z = self.fc_latent(h)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        h = self.fc_decode(z)
        h = h.view(z.shape[0], self.encoded_channels, self.encoded_len)
        logits = self.decoder_conv(h)
        # Safety crop/pad in case a future seq_len is not divisible by 2**n_conv_blocks.
        if logits.shape[-1] > self.seq_len:
            logits = logits[..., : self.seq_len]
        elif logits.shape[-1] < self.seq_len:
            pad_len = self.seq_len - logits.shape[-1]
            logits = F.pad(logits, (0, pad_len))
        return logits

    def forward(self, x_tokens: torch.Tensor) -> torch.Tensor:
        z = self.encode(x_tokens)
        return self.decode(z)

## Training and embeddings

In [5]:
def train_autoencoder(
    X_healthy: np.ndarray,
    groups: List[str],
    params: Dict,
    epochs: int,
    batch_size: int,
    seed: int,
    device: torch.device,
    verbose: bool = False,
) -> Tuple[HypnoCNNAutoencoder, Dict[str, object]]:
    set_global_seed(seed)

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx, val_idx = next(splitter.split(X_healthy, groups=groups))

    train_ds = HypnogramDataset(X_healthy[train_idx])
    val_ds = HypnogramDataset(X_healthy[val_idx])
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    model = HypnoCNNAutoencoder(
        seq_len=SEQ_LEN,
        num_classes=NUM_CLASSES,
        latent_dim=params["latent_dim"],
        n_conv_blocks=params["n_conv_blocks"],
        channels=params["filters_base"],
        kernel_size=params["kernel_size"],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"], weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_CLASS)

    best_state = None
    best_val_loss = float("inf")
    patience = 10
    min_delta = 1e-4
    patience_counter = 0
    train_losses: List[float] = []
    val_losses: List[float] = []

    for epoch in range(int(epochs)):
        model.train()
        running = []
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch)              # [B, C, L]
            loss = criterion(logits, batch)    # target [B, L]
            loss.backward()
            optimizer.step()
            running.append(float(loss.detach().cpu()))

        train_loss = float(np.mean(running)) if running else np.nan
        train_losses.append(train_loss)

        model.eval()
        running_val = []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                logits = model(batch)
                loss = criterion(logits, batch)
                running_val.append(float(loss.detach().cpu()))
        val_loss = float(np.mean(running_val)) if running_val else np.nan
        val_losses.append(val_loss)

        if verbose:
            print(f"epoch={epoch+1:03d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    hist = {
        "best_val_loss": float(best_val_loss),
        "final_train_loss": float(train_losses[-1]) if train_losses else np.nan,
        "epochs_run": len(train_losses),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_idx": train_idx.tolist(),
        "val_idx": val_idx.tolist(),
    }
    return model, hist


def encode_records(model: HypnoCNNAutoencoder, X: np.ndarray, batch_size: int, device: torch.device) -> np.ndarray:
    ds = HypnogramDataset(X)
    loader = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    Zs = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            z = model.encode(batch).detach().cpu().numpy()
            Zs.append(z)
    return np.concatenate(Zs, axis=0)

## UMAP, clustering, internal HPO scores


In [ ]:
def get_umap_seed(args) -> int:
    """Use one fixed UMAP seed across all HPO trials for fair geometry comparison."""
    return int(getattr(args, "umap_seed", args.seed))


def fit_umap(
    Z_healthy: np.ndarray,
    seed: int,
    n_neighbors: int = 15,
    n_components: int = 2,
    min_dist: float = 0.1,
) -> Tuple[object, np.ndarray, StandardScaler]:
    scaler = StandardScaler()
    Z_scaled = scaler.fit_transform(Z_healthy)

    # Safety for very small subsets. UMAP requires n_neighbors < n_samples in practice.
    n_neighbors_safe = int(min(n_neighbors, max(2, len(Z_scaled) - 1)))
    n_components_safe = int(max(2, n_components))

    reducer = umap.UMAP(
        n_components=n_components_safe,
        n_neighbors=n_neighbors_safe,
        min_dist=float(min_dist),
        metric="euclidean",
        random_state=int(seed),
    )
    U = reducer.fit_transform(Z_scaled)
    return reducer, U, scaler


def transform_umap(reducer: object, scaler: StandardScaler, Z: np.ndarray) -> np.ndarray:
    return reducer.transform(scaler.transform(Z))


def fit_clusterer(U: np.ndarray, method: str, n_clusters: int, seed: int):
    if method == "kmeans":
        model = KMeans(
            n_clusters=n_clusters,
            init="k-means++",
            n_init=50,
            random_state=seed,
        )
        labels = model.fit_predict(U)
        return model, labels.astype(int)
    if method == "gmm":
        model = GaussianMixture(
            n_components=n_clusters,
            covariance_type="full",
            n_init=10,
            random_state=seed,
        )
        labels = model.fit_predict(U)
        return model, labels.astype(int)
    if method == "agglomerative":
        model = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
        labels = model.fit_predict(U)
        return model, labels.astype(int)
    raise ValueError(f"Unknown clustering method: {method}")


def predict_cluster_labels(U_new: np.ndarray, U_ref: np.ndarray, labels_ref: np.ndarray, method: str, clusterer) -> np.ndarray:
    if len(U_new) == 0:
        return np.array([], dtype=int)
    if method in {"kmeans", "gmm"} and hasattr(clusterer, "predict"):
        return clusterer.predict(U_new).astype(int)

    # Agglomerative has no predict(). Assign by nearest reference cluster centroid.
    centroids = []
    cluster_ids = sorted(np.unique(labels_ref))
    for cid in cluster_ids:
        centroids.append(U_ref[labels_ref == cid].mean(axis=0))
    centroids = np.vstack(centroids)
    dists = ((U_new[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    nearest = np.argmin(dists, axis=1)
    return np.asarray([cluster_ids[i] for i in nearest], dtype=int)


def valid_cluster_labels(U: np.ndarray, labels: np.ndarray) -> bool:
    if len(U) < 3 or len(np.unique(labels)) < 2:
        return False
    counts = pd.Series(labels).value_counts()
    return bool(counts.min() >= 2)


def safe_silhouette(U: np.ndarray, labels: np.ndarray) -> float:
    if not valid_cluster_labels(U, labels):
        return float("nan")
    return float(silhouette_score(U, labels))


def safe_davies_bouldin(U: np.ndarray, labels: np.ndarray) -> float:
    if not valid_cluster_labels(U, labels):
        return float("nan")
    return float(davies_bouldin_score(U, labels))


def objective_from_silhouette_and_dbi(silhouette: float, dbi: float) -> float:
    """Composite internal clustering objective. Higher is better.

    Silhouette is normalized from [-1, 1] to [0, 1].
    Davies-Bouldin is transformed from lower-is-better to higher-is-better via 1/(1+DBI).
    """
    if not (np.isfinite(silhouette) and np.isfinite(dbi)):
        return -999.0
    silhouette_norm = (float(silhouette) + 1.0) / 2.0
    dbi_score = 1.0 / (1.0 + float(dbi))
    return float(OBJECTIVE_SILHOUETTE_WEIGHT * silhouette_norm + OBJECTIVE_DBI_WEIGHT * dbi_score)


## HPO

In [7]:
def sample_params(trial: optuna.Trial) -> Dict:
    return {
        "latent_dim": trial.suggest_categorical("latent_dim", LATENT_DIM_CHOICES),
        "n_conv_blocks": trial.suggest_categorical("n_conv_blocks", N_CONV_BLOCKS_CHOICES),
        "filters_base": trial.suggest_categorical("filters_base", FILTERS_BASE_CHOICES),
        "kernel_size": trial.suggest_categorical("kernel_size", KERNEL_SIZE_CHOICES),
        "learning_rate": trial.suggest_float("learning_rate", LEARNING_RATE_LOW, LEARNING_RATE_HIGH, log=True),
        "n_clusters": trial.suggest_categorical("n_clusters", N_CLUSTERS_CHOICES),
        "umap_n_neighbors": trial.suggest_categorical("umap_n_neighbors", UMAP_N_NEIGHBORS_CHOICES),
        "umap_n_components": trial.suggest_categorical("umap_n_components", UMAP_N_COMPONENTS_CHOICES),
        "umap_min_dist": trial.suggest_categorical("umap_min_dist", UMAP_MIN_DIST_CHOICES),
    }


def make_objective(method: str, X_healthy: np.ndarray, groups: List[str], args, device: torch.device):
    def objective(trial: optuna.Trial) -> float:
        params = sample_params(trial)

        # Model/clustering seed can vary by trial; UMAP seed is fixed across trials.
        trial_seed = int(args.seed + 1000 * VALID_CLUSTER_METHODS.index(method) + trial.number)
        umap_seed = get_umap_seed(args)

        try:
            model, hist = train_autoencoder(
                X_healthy=X_healthy,
                groups=groups,
                params=params,
                epochs=args.epochs,
                batch_size=args.batch_size,
                seed=trial_seed,
                device=device,
                verbose=False,
            )
            Z_all = encode_records(model, X_healthy, batch_size=args.batch_size, device=device)
            train_idx = np.asarray(hist["train_idx"], dtype=int)
            val_idx = np.asarray(hist["val_idx"], dtype=int)
            Z_train = Z_all[train_idx]
            Z_val = Z_all[val_idx]

            reducer, U_train, scaler = fit_umap(
                Z_train,
                seed=umap_seed,
                n_neighbors=params["umap_n_neighbors"],
                n_components=params["umap_n_components"],
                min_dist=params["umap_min_dist"],
            )
            U_val = transform_umap(reducer, scaler, Z_val) if len(Z_val) else np.empty((0, U_train.shape[1]))

            clusterer, labels_train = fit_clusterer(
                U_train,
                method=method,
                n_clusters=params["n_clusters"],
                seed=trial_seed,
            )
            labels_val = predict_cluster_labels(U_val, U_train, labels_train, method=method, clusterer=clusterer)

            sil = safe_silhouette(U_val, labels_val)
            dbi = safe_davies_bouldin(U_val, labels_val)
            objective_value = objective_from_silhouette_and_dbi(sil, dbi)

            cluster_counts = pd.Series(labels_train).value_counts().sort_index().to_dict()
            trial.set_user_attr("objective_composite", float(objective_value))
            trial.set_user_attr("silhouette", float(sil) if np.isfinite(sil) else None)
            trial.set_user_attr("davies_bouldin", float(dbi) if np.isfinite(dbi) else None)
            trial.set_user_attr("umap_seed", int(umap_seed))
            trial.set_user_attr("trial_seed", int(trial_seed))
            trial.set_user_attr("best_val_loss", float(hist["best_val_loss"]))
            trial.set_user_attr("epochs_run", int(hist["epochs_run"]))
            trial.set_user_attr("cluster_counts", {str(k): int(v) for k, v in cluster_counts.items()})
            for k, v in params.items():
                trial.set_user_attr(k, float(v) if isinstance(v, np.floating) else v)
            return float(objective_value)
        except Exception as exc:
            trial.set_user_attr("error", repr(exc))
            return -999.0

    return objective


def trial_dataframe(study: optuna.Study) -> pd.DataFrame:
    rows = []
    for t in study.trials:
        rows.append({
            "number": t.number,
            "value": t.value,
            "state": str(t.state),
            **t.params,
            **t.user_attrs,
        })
    return pd.DataFrame(rows)

## Plots and reporting

In [8]:
def save_loss_curve(hist: Dict[str, object], out_dir: Path) -> None:
    plt.figure(figsize=(8, 5))
    plt.plot(hist["train_losses"], label="train")
    plt.plot(hist["val_losses"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("CrossEntropyLoss")
    plt.title("Autoencoder training loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "final_loss_curve.png", dpi=200)
    plt.close()


def save_umap_plots(
    U_healthy: np.ndarray,
    labels_healthy: np.ndarray,
    U_st: np.ndarray,
    labels_st: np.ndarray,
    out_dir: Path,
) -> None:
    plt.figure(figsize=(7, 6))
    sc = plt.scatter(U_healthy[:, 0], U_healthy[:, 1], c=labels_healthy, s=35, alpha=0.85)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("Healthy reference UMAP colored by cluster")
    plt.colorbar(sc, label="cluster")
    plt.tight_layout()
    plt.savefig(out_dir / "final_umap_healthy_by_cluster.png", dpi=200)
    plt.close()

    plt.figure(figsize=(7, 6))
    plt.scatter(U_healthy[:, 0], U_healthy[:, 1], s=35, alpha=0.85, label="SC healthy reference")
    if len(U_st):
        plt.scatter(U_st[:, 0], U_st[:, 1], s=35, alpha=0.85, marker="x", label="ST projected")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("UMAP projection colored by cohort")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "final_umap_by_cohort_projection.png", dpi=200)
    plt.close()

    plt.figure(figsize=(7, 6))
    U_all = np.vstack([U_healthy, U_st]) if len(U_st) else U_healthy
    labels_all = np.concatenate([labels_healthy, labels_st]) if len(U_st) else labels_healthy
    sc = plt.scatter(U_all[:, 0], U_all[:, 1], c=labels_all, s=35, alpha=0.85)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title("UMAP projection colored by assigned cluster")
    plt.colorbar(sc, label="cluster")
    plt.tight_layout()
    plt.savefig(out_dir / "final_umap_combined_by_assigned_cluster.png", dpi=200)
    plt.close()



def infer_st_night_digit(record_id: str) -> str:
    """Return the ST night digit directly after the subject id.

    Example: ST7011J0 -> subject ST701, night digit 1.
    """
    rid = str(record_id).upper()
    return rid[5] if rid.startswith("ST") and len(rid) > 5 else ""


ST_TREATMENT_NIGHT_BY_SUBJECT = {
    # Sleep Telemetry table: subject Nr -> placebo/temazepam night number.
    1: {"Placebo": "1", "Temazepam": "2"},
    2: {"Placebo": "2", "Temazepam": "1"},
    4: {"Placebo": "1", "Temazepam": "2"},
    5: {"Placebo": "2", "Temazepam": "1"},
    6: {"Placebo": "1", "Temazepam": "2"},
    7: {"Placebo": "1", "Temazepam": "2"},
    8: {"Placebo": "2", "Temazepam": "1"},
    9: {"Placebo": "2", "Temazepam": "1"},
    10: {"Placebo": "1", "Temazepam": "2"},
    11: {"Placebo": "2", "Temazepam": "1"},
    12: {"Placebo": "1", "Temazepam": "2"},
    13: {"Placebo": "2", "Temazepam": "1"},
    14: {"Placebo": "1", "Temazepam": "2"},
    15: {"Placebo": "1", "Temazepam": "2"},
    16: {"Placebo": "2", "Temazepam": "1"},
    17: {"Placebo": "1", "Temazepam": "2"},
    18: {"Placebo": "2", "Temazepam": "1"},
    19: {"Placebo": "2", "Temazepam": "1"},
    20: {"Placebo": "1", "Temazepam": "2"},
    21: {"Placebo": "2", "Temazepam": "1"},
    22: {"Placebo": "1", "Temazepam": "2"},
    24: {"Placebo": "1", "Temazepam": "2"},
}


def infer_st_subject_nr(record_id: str) -> Optional[int]:
    """Map Sleep-EDF ST record id to the table subject number.

    Example: ST7011J0 -> ST701 -> subject Nr 1.
    """
    rid = str(record_id).upper()
    m = re.match(r"ST7(\d{2})", rid)
    if not m:
        return None
    return int(m.group(1))


def infer_st_treatment(record_id: str, args=None) -> str:
    """Infer Placebo/Temazepam from the exact Sleep Telemetry subject-night table."""
    digit = infer_st_night_digit(record_id)
    subject_nr = infer_st_subject_nr(record_id)
    mapping = ST_TREATMENT_NIGHT_BY_SUBJECT.get(subject_nr)
    if mapping is not None:
        if digit == mapping["Placebo"]:
            return "Placebo"
        if digit == mapping["Temazepam"]:
            return "Temazepam"
        return "Unknown"

    # Fallback only for records not contained in the supplied table.
    placebo_digit = str(getattr(args, "st_placebo_night_digit", "1")) if args is not None else "1"
    temazepam_digit = str(getattr(args, "st_temazepam_night_digit", "2")) if args is not None else "2"
    if digit == placebo_digit:
        return "Placebo"
    if digit == temazepam_digit:
        return "Temazepam"
    return "Unknown"


def stage_distribution_from_sequence(seq: np.ndarray) -> Dict[str, float]:
    """Post-hoc visualization values only: % of valid loaded hypnogram sequence.

    These values are not used for autoencoder training, UMAP fitting, clustering,
    or the HPO objective.
    """
    arr = np.asarray(seq, dtype=int)
    valid = arr[np.isin(arr, SLEEP_STAGE_CLASSES)]
    if len(valid) == 0:
        return {"Wake": np.nan, "Light (N1+N2)": np.nan, "Deep (N3)": np.nan, "REM": np.nan}
    denom = float(len(valid))
    return {
        "Wake": 100.0 * float(np.sum(valid == STAGE_TO_CLASS["W"])) / denom,
        "Light (N1+N2)": 100.0 * float(np.sum(np.isin(valid, [STAGE_TO_CLASS["N1"], STAGE_TO_CLASS["N2"]]))) / denom,
        "Deep (N3)": 100.0 * float(np.sum(valid == STAGE_TO_CLASS["N3"])) / denom,
        "REM": 100.0 * float(np.sum(valid == STAGE_TO_CLASS["REM"])) / denom,
    }


def make_stage_distribution_dataframe(
    records: List[NightRecord],
    X: np.ndarray,
    labels: np.ndarray,
    args=None,
) -> pd.DataFrame:
    rows = []
    for rec, seq, lab in zip(records, X, labels):
        dist = stage_distribution_from_sequence(seq)
        treatment = infer_st_treatment(rec.record_id, args) if rec.cohort == "ST" and args is not None else "Healthy reference"
        for stage_name, pct in dist.items():
            rows.append({
                "record_id": rec.record_id,
                "subject_id": rec.subject_id,
                "cohort": rec.cohort,
                "treatment": treatment,
                "cluster": int(lab),
                "sleep_stage": stage_name,
                "percentage": float(pct),
            })
    return pd.DataFrame(rows)


def save_stage_distribution_boxplot(stage_df: pd.DataFrame, out_dir: Path) -> None:
    """Boxplot like: Wake / Light / Deep / REM vs percentage, colored by cluster."""
    if stage_df.empty:
        return
    stage_order = ["Wake", "Light (N1+N2)", "Deep (N3)", "REM"]
    clusters = sorted(stage_df["cluster"].dropna().astype(int).unique().tolist())
    if not clusters:
        return

    fig, ax = plt.subplots(figsize=(12, 7))
    width = 0.8 / max(1, len(clusters))
    positions = []
    data = []
    colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", [])

    for si, stage in enumerate(stage_order):
        for ci, cluster in enumerate(clusters):
            vals = stage_df[(stage_df["sleep_stage"] == stage) & (stage_df["cluster"] == cluster)]["percentage"].dropna().values
            if len(vals) == 0:
                vals = np.array([np.nan])
            data.append(vals)
            positions.append(si + (ci - (len(clusters) - 1) / 2.0) * width)

    bp = ax.boxplot(
        data,
        positions=positions,
        widths=width * 0.85,
        patch_artist=True,
        manage_ticks=False,
        showfliers=False,
    )
    for i, patch in enumerate(bp["boxes"]):
        cluster = clusters[i % len(clusters)]
        patch.set_facecolor(colors[cluster % len(colors)] if colors else None)
        patch.set_alpha(0.85)
    for median in bp["medians"]:
        median.set_color("black")
        median.set_linewidth(1.2)

    handles = []
    for cluster in clusters:
        color = colors[cluster % len(colors)] if colors else "gray"
        handles.append(plt.Line2D([0], [0], marker="s", linestyle="", markersize=10, markerfacecolor=color, markeredgecolor=color, label=str(cluster)))

    ax.set_xticks(range(len(stage_order)))
    ax.set_xticklabels(["% Wake", "% Light (N1+N2)", "% Deep (N3)", "% REM"])
    ax.set_ylabel("Percentage of Loaded Valid Sequence (%)")
    ax.set_xlabel("Sleep Stage")
    ax.set_title("Sleep Stage Distribution by Cluster Including Assigned ST Nights")
    ax.legend(handles=handles, title="Cluster ID", loc="best")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_dir / "final_sleep_stage_distribution_by_cluster.png", dpi=200)
    plt.close(fig)


def save_st_treatment_cluster_distribution(st_df: pd.DataFrame, out_dir: Path) -> None:
    """Grouped bar plot: placebo vs temazepam cluster assignment percentages."""
    if st_df.empty or "treatment" not in st_df.columns:
        return
    night_df = st_df[["record_id", "subject_id", "treatment", "cluster"]].drop_duplicates()
    night_df = night_df[night_df["treatment"].isin(["Placebo", "Temazepam"])]
    if night_df.empty:
        return

    counts = pd.crosstab(night_df["cluster"], night_df["treatment"])
    for col in ["Placebo", "Temazepam"]:
        if col not in counts.columns:
            counts[col] = 0
    counts = counts[["Placebo", "Temazepam"]].sort_index()
    percentages = counts.div(counts.sum(axis=0).replace(0, np.nan), axis=1) * 100.0
    percentages.to_csv(out_dir / "final_st_treatment_cluster_distribution.csv")

    ax = percentages.plot(kind="bar", figsize=(9, 5), rot=0)
    ax.set_xlabel("Assigned Cluster")
    ax.set_ylabel("Nights assigned to cluster (%)")
    ax.set_title("ST Cluster Assignment: Placebo vs Temazepam")
    ax.legend(title="Treatment")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(out_dir / "final_st_treatment_cluster_distribution.png", dpi=200)
    plt.close()


def save_st_drug_trial_umap_plot(
    U_healthy: np.ndarray,
    labels_healthy: np.ndarray,
    U_st: np.ndarray,
    labels_st: np.ndarray,
    records_st: List[NightRecord],
    args,
    out_dir: Path,
) -> None:
    """UMAP plot with paired arrows Placebo -> Temazepam for ST subjects."""
    if len(U_st) == 0 or len(records_st) == 0:
        return

    st_points = pd.DataFrame({
        "record_id": [r.record_id for r in records_st],
        "subject_id": [r.subject_id for r in records_st],
        "treatment": [infer_st_treatment(r.record_id, args) for r in records_st],
        "cluster": labels_st.astype(int),
        "umap1": U_st[:, 0],
        "umap2": U_st[:, 1],
    })
    st_points.to_csv(out_dir / "final_st_drug_trial_umap_points.csv", index=False)
    paired = st_points[st_points["treatment"].isin(["Placebo", "Temazepam"])]
    if paired.empty:
        return

    fig, ax = plt.subplots(figsize=(11, 7))
    # Faint healthy reference background, colored by discovered cluster.
    ax.scatter(U_healthy[:, 0], U_healthy[:, 1], c=labels_healthy, s=20, alpha=0.12, label="SC reference")

    placebo = paired[paired["treatment"] == "Placebo"]
    temazepam = paired[paired["treatment"] == "Temazepam"]
    ax.scatter(placebo["umap1"], placebo["umap2"], marker="X", s=90, facecolors="none", edgecolors="black", linewidths=1.6, label="Placebo Night")
    ax.scatter(temazepam["umap1"], temazepam["umap2"], marker="X", s=90, c=temazepam["cluster"], linewidths=1.6, label="Temazepam Night")

    arrow_count = 0
    for subject_id, grp in paired.groupby("subject_id"):
        p = grp[grp["treatment"] == "Placebo"]
        t = grp[grp["treatment"] == "Temazepam"]
        if len(p) and len(t):
            p0 = p.iloc[0]
            t0 = t.iloc[0]
            ax.annotate(
                "",
                xy=(t0["umap1"], t0["umap2"]),
                xytext=(p0["umap1"], p0["umap2"]),
                arrowprops=dict(arrowstyle="->", lw=1.4, alpha=0.75),
            )
            ax.text(t0["umap1"], t0["umap2"], f" {subject_id}", fontsize=8, weight="bold", alpha=0.8)
            arrow_count += 1

    ax.set_xlabel("UMAP Dimension 1")
    ax.set_ylabel("UMAP Dimension 2")
    ax.set_title("Drug Trial Tracking: Placebo → Temazepam")
    ax.grid(alpha=0.25)
    ax.legend(title="Treatment Indicator", loc="best")
    fig.tight_layout()
    fig.savefig(out_dir / "final_st_drug_trial_umap_placebo_to_temazepam.png", dpi=200)
    plt.close(fig)

    with open(out_dir / "final_st_drug_trial_pairs_summary.json", "w") as f:
        json.dump({"n_st_points": int(len(st_points)), "n_placebo_temazepam_pairs_with_arrows": int(arrow_count)}, f, indent=2)


def save_dendrogram(U_healthy: np.ndarray, out_dir: Path) -> None:
    if len(U_healthy) < 3:
        return
    Z = linkage(U_healthy, method="ward")
    plt.figure(figsize=(12, 6))
    dendrogram(Z, no_labels=True, color_threshold=None)
    plt.title("Hierarchical dendrogram on healthy UMAP embedding")
    plt.xlabel("Healthy reference nights")
    plt.ylabel("Ward distance")
    plt.tight_layout()
    plt.savefig(out_dir / "final_healthy_dendrogram_umap.png", dpi=200)
    plt.close()


def final_run_for_method(
    method: str,
    best_params: Dict,
    records_healthy: List[NightRecord],
    records_st: List[NightRecord],
    X_healthy: np.ndarray,
    X_st: np.ndarray,
    groups: List[str],
    args,
    device: torch.device,
    method_out_dir: Path,
) -> Dict[str, object]:
    method_out_dir.mkdir(parents=True, exist_ok=True)
    seed = int(args.seed + 999)
    umap_seed = get_umap_seed(args)

    final_epochs = args.final_epochs if args.final_epochs is not None else args.epochs
    model, hist = train_autoencoder(
        X_healthy=X_healthy,
        groups=groups,
        params=best_params,
        epochs=final_epochs,
        batch_size=args.batch_size,
        seed=seed,
        device=device,
        verbose=True,
    )

    Z_healthy = encode_records(model, X_healthy, batch_size=args.batch_size, device=device)
    Z_st = encode_records(model, X_st, batch_size=args.batch_size, device=device) if len(X_st) else np.empty((0, best_params["latent_dim"]))

    reducer, U_healthy, scaler = fit_umap(
        Z_healthy,
        seed=umap_seed,
        n_neighbors=best_params["umap_n_neighbors"],
        n_components=best_params["umap_n_components"],
        min_dist=best_params["umap_min_dist"],
    )
    U_st = transform_umap(reducer, scaler, Z_st) if len(Z_st) else np.empty((0, 2))

    clusterer, labels_healthy = fit_clusterer(U_healthy, method=method, n_clusters=best_params["n_clusters"], seed=seed)
    labels_st = predict_cluster_labels(U_st, U_healthy, labels_healthy, method=method, clusterer=clusterer)

    sil = safe_silhouette(U_healthy, labels_healthy)
    dbi = safe_davies_bouldin(U_healthy, labels_healthy)
    objective_composite = objective_from_silhouette_and_dbi(sil, dbi)
    # Save model-related artefacts separately from plots.
    model_dir = method_out_dir / "model"
    plots_dir = method_out_dir / "plots"
    model_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    torch.save({
        "model_state_dict": model.state_dict(),
        "params": best_params,
        "seq_len": SEQ_LEN,
        "num_classes": NUM_CLASSES,
        "channels": best_params["filters_base"],
        "kernel_size": best_params["kernel_size"],
        "weight_decay": WEIGHT_DECAY,
        "umap_seed": umap_seed,
        "umap_n_neighbors": best_params["umap_n_neighbors"],
        "umap_n_components": best_params["umap_n_components"],
        "umap_min_dist": best_params["umap_min_dist"],
    }, model_dir / "final_autoencoder.pt")
    np.save(model_dir / "final_healthy_latent_vectors.npy", Z_healthy)
    np.save(model_dir / "final_st_latent_vectors.npy", Z_st)
    np.save(model_dir / "final_healthy_umap.npy", U_healthy)
    np.save(model_dir / "final_st_umap_projected.npy", U_st)
    np.save(model_dir / "final_healthy_cluster_labels.npy", labels_healthy)
    np.save(model_dir / "final_st_assigned_cluster_labels.npy", labels_st)

    # Result tables
    healthy_rows = []
    for rec, lab in zip(records_healthy, labels_healthy):
        row = {
            "record_id": rec.record_id,
            "subject_id": rec.subject_id,
            "cohort": rec.cohort,
            "cluster": int(lab),
            "path": rec.path,
        }
        healthy_rows.append(row)
    healthy_df = pd.DataFrame(healthy_rows)
    healthy_df.to_csv(method_out_dir / "final_healthy_night_level_results.csv", index=False)

    st_rows = []
    for rec, lab in zip(records_st, labels_st):
        row = {
            "record_id": rec.record_id,
            "subject_id": rec.subject_id,
            "st_subject_nr": infer_st_subject_nr(rec.record_id),
            "night_digit": infer_st_night_digit(rec.record_id),
            "treatment": infer_st_treatment(rec.record_id, args),
            "cohort": rec.cohort,
            "assigned_cluster": int(lab),
            "path": rec.path,
        }
        st_rows.append(row)
    st_df = pd.DataFrame(st_rows)
    st_df.to_csv(method_out_dir / "final_st_projected_night_level_results.csv", index=False)

    # Post-hoc visualization only. These stage percentages are not used for HPO,
    # representation learning, UMAP fitting, or cluster fitting.
    records_all = records_healthy + records_st
    X_all = np.vstack([X_healthy, X_st]) if len(X_st) else X_healthy
    labels_all = np.concatenate([labels_healthy, labels_st]) if len(labels_st) else labels_healthy
    stage_df = make_stage_distribution_dataframe(records_all, X_all, labels_all, args=args)
    stage_df.to_csv(plots_dir / "final_sleep_stage_distribution_by_cluster.csv", index=False)
    save_stage_distribution_boxplot(stage_df, plots_dir)
    save_st_treatment_cluster_distribution(stage_df[stage_df["cohort"] == "ST"], plots_dir)
    save_st_drug_trial_umap_plot(U_healthy, labels_healthy, U_st, labels_st, records_st, args, plots_dir)

    stability_runs = int(getattr(args, "stability_runs", 0))
    stability_info = {}
    if stability_runs > 1:
        seeds = [seed + i for i in range(stability_runs)]
        labels_list = [labels_healthy]
        for seed_i in seeds[1:]:
            model_i, _ = train_autoencoder(
                X_healthy=X_healthy,
                groups=groups,
                params=best_params,
                epochs=final_epochs,
                batch_size=args.batch_size,
                seed=seed_i,
                device=device,
                verbose=False,
            )
            Z_i = encode_records(model_i, X_healthy, batch_size=args.batch_size, device=device)
            reducer_i, U_i, scaler_i = fit_umap(
                Z_i,
                seed=umap_seed,
                n_neighbors=best_params["umap_n_neighbors"],
                n_components=best_params["umap_n_components"],
                min_dist=best_params["umap_min_dist"],
            )
            clusterer_i, labels_i = fit_clusterer(U_i, method=method, n_clusters=best_params["n_clusters"], seed=seed_i)
            labels_list.append(labels_i)

        aris = [float(adjusted_rand_score(labels_list[0], labs)) for labs in labels_list[1:]]
        stability_info = {
            "seeds": seeds,
            "ari_vs_seed0": aris,
            "ari_mean": float(np.mean(aris)) if aris else float("nan"),
            "ari_min": float(np.min(aris)) if aris else float("nan"),
        }

    # Plots
    save_loss_curve(hist, plots_dir)
    save_umap_plots(U_healthy, labels_healthy, U_st, labels_st, plots_dir)
    save_dendrogram(U_healthy, plots_dir)

    report = {
        "method": method,
        "best_params": best_params,
        "fixed_model_params": {
            "seq_len": SEQ_LEN,
            "channels": best_params["filters_base"],
            "kernel_size": best_params["kernel_size"],
            "weight_decay": WEIGHT_DECAY,
            "optimizer": "Adam",
            "loss": "CrossEntropyLoss(ignore_index=5)",
            "umap_n_neighbors": best_params["umap_n_neighbors"],
            "umap_n_components": best_params["umap_n_components"],
            "umap_min_dist": best_params["umap_min_dist"],
            "umap_seed": umap_seed,
            "kmeans_init": "k-means++" if method == "kmeans" else None,
        },
        "training_summary": {
            "best_val_loss": hist["best_val_loss"],
            "final_train_loss": hist["final_train_loss"],
            "epochs_run": hist["epochs_run"],
        },
        "final_objective_composite_healthy_reference": objective_composite,
        "final_silhouette_healthy_reference": sil,
        "final_davies_bouldin_healthy_reference": dbi,
        "stability": stability_info,
        "healthy_cluster_counts": {str(k): int(v) for k, v in pd.Series(labels_healthy).value_counts().sort_index().items()},
        "st_assigned_cluster_counts": {str(k): int(v) for k, v in pd.Series(labels_st).value_counts().sort_index().items()},
        "n_healthy_records": len(records_healthy),
        "n_st_records": len(records_st),
        "output_dirs": {
            "method_dir": str(method_out_dir),
            "model_dir": str(model_dir),
            "plots_dir": str(plots_dir),
        },
        "note": "This HPO script uses only hypnogram stage sequences as model input. The objective uses only internal cluster validity: normalized Silhouette and transformed Davies-Bouldin on SC/healthy reference clusters. Sleep-stage percentage summaries are saved only as post-hoc plots/tables after final clustering; they are not used in HPO, representation learning, UMAP fitting, or cluster fitting.",
    }
    with open(method_out_dir / "final_report.json", "w") as f:
        json.dump(report, f, indent=2)

    return report


## Main

In [9]:
def parse_cluster_methods(value: str) -> List[str]:
    methods = [v.strip().lower() for v in value.split(",") if v.strip()]
    invalid = [m for m in methods if m not in VALID_CLUSTER_METHODS]
    if invalid:
        raise argparse.ArgumentTypeError(f"Invalid cluster method(s): {invalid}. Valid: {VALID_CLUSTER_METHODS}")
    return methods


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Healthy-reference CNN autoencoder HPO for Sleep-EDF preprocessed FIF hypnograms")
    p.add_argument("--data_dir", type=str, required=True, help="Root folder containing sleep-cassette/preprocessed and sleep-telemetry/preprocessed FIF files")
    p.add_argument("--out_dir", type=str, required=True, help="Output directory")
    p.add_argument("--cluster_methods", type=parse_cluster_methods, default=parse_cluster_methods("kmeans,gmm,agglomerative"),
                   help="Comma-separated methods: kmeans,gmm,agglomerative")
    p.add_argument("--n_trials", type=int, default=30, help="Optuna trials per clustering method")
    p.add_argument("--epochs", type=int, default=30, help="Max epochs per HPO trial")
    p.add_argument("--final_epochs", type=int, default=None, help="Max epochs for final retraining; defaults to --epochs")
    p.add_argument("--batch_size", type=int, default=32, help="Batch size")
    p.add_argument("--seed", type=int, default=42, help="Random seed")
    p.add_argument("--study_name", type=str, default="sleepedf_fif_healthy_reference_hpo")
    p.add_argument("--storage", type=str, default=None, help="Optional Optuna storage URL, e.g. sqlite:///study.db")
    p.add_argument("--n_jobs", type=int, default=1, help="Parallel Optuna jobs. Keep 1 for GPU stability.")
    p.add_argument("--umap_seed", type=int, default=None, help="Fixed UMAP random seed. Defaults to --seed if omitted.")
    p.add_argument("--stability_runs", type=int, default=3, help="Number of stability runs for ARI; set 0 or 1 to skip.")
    p.add_argument("--st_placebo_night_digit", type=str, default="1", help="Sleep Telemetry night digit interpreted as Placebo")
    p.add_argument("--st_temazepam_night_digit", type=str, default="2", help="Sleep Telemetry night digit interpreted as Temazepam")
    return p.parse_args()


def run_pipeline(args) -> None:
    set_global_seed(args.seed)
    device = get_device()

    # All generated artefacts are written into one dedicated results folder.
    # If args.out_dir is the project folder, results go to <project>/hpo_results.
    # If args.out_dir already points to hpo_results, it is used directly.
    base_out_dir = Path(args.out_dir)
    out_dir = base_out_dir if base_out_dir.name == "hpo_results" else base_out_dir / "hpo_results"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Keep the Optuna SQLite study inside hpo_results as well.
    # Example: storage="sqlite:///study.db" -> sqlite:///<out_dir>/study.db
    storage = getattr(args, "storage", None)
    if storage:
        storage = str(storage)
        sqlite_prefix = "sqlite:///"
        if storage.startswith(sqlite_prefix):
            sqlite_path = Path(storage[len(sqlite_prefix):])
            if not sqlite_path.is_absolute():
                storage = sqlite_prefix + (out_dir / sqlite_path).as_posix()

    print(f"Device: {device}")
    print(f"Writing all outputs to: {out_dir}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    print(f"Loading preprocessed FIF records from: {args.data_dir}")
    records = load_records(Path(args.data_dir))
    records_healthy = [r for r in records if r.cohort == "SC"]
    records_st = [r for r in records if r.cohort == "ST"]

    if len(records_healthy) < 10:
        raise RuntimeError(f"Too few SC/healthy records found: {len(records_healthy)}")

    print(f"Loaded records: total={len(records)}, SC={len(records_healthy)}, ST={len(records_st)}")

    metadata = pd.DataFrame([{
        "record_id": r.record_id,
        "subject_id": r.subject_id,
        "cohort": r.cohort,
        "n_epochs_full": len(r.stages),
        "duration_hours_full": len(r.stages) * EPOCH_SEC / 3600.0,
        "path": r.path,
    } for r in records])
    metadata.to_csv(out_dir / "loaded_records_metadata.csv", index=False)

    X_healthy = records_to_array(records_healthy, seq_len=SEQ_LEN)
    X_st = records_to_array(records_st, seq_len=SEQ_LEN) if records_st else np.empty((0, SEQ_LEN), dtype=np.int64)
    groups = [r.subject_id for r in records_healthy]

    config = {
        "args": {**vars(args), "cluster_methods": args.cluster_methods},
        "hpo_search_space": {
            "latent_dim": LATENT_DIM_CHOICES,
            "n_conv_blocks": N_CONV_BLOCKS_CHOICES,
            "filters_base": FILTERS_BASE_CHOICES,
            "kernel_size": KERNEL_SIZE_CHOICES,
            "learning_rate": [LEARNING_RATE_LOW, LEARNING_RATE_HIGH, "log"],
            "n_clusters": N_CLUSTERS_CHOICES,
            "umap_n_neighbors": UMAP_N_NEIGHBORS_CHOICES,
            "umap_n_components": UMAP_N_COMPONENTS_CHOICES,
            "umap_min_dist": UMAP_MIN_DIST_CHOICES,
        },
        "fixed_params": {
            "seq_len": SEQ_LEN,
            "weight_decay": WEIGHT_DECAY,
            "optimizer": "Adam",
            "batch_size": args.batch_size,
            "umap_seed": get_umap_seed(args),
            "hpo_objective": "0.70 * normalized_silhouette + 0.30 * transformed_davies_bouldin",
            "sleep_posthoc_metrics": "computed only after final clustering for visualization; not used in HPO objective",
            "output_layout": "per method: model/ contains .pt and .npy artefacts; plots/ contains .png and plot source tables",
            "st_treatment_mapping": "uses explicit Sleep Telemetry subject-night table; CLI night digits are fallback only",
            "st_placebo_night_digit": str(getattr(args, "st_placebo_night_digit", "1")),
            "st_temazepam_night_digit": str(getattr(args, "st_temazepam_night_digit", "2")),
            "kmeans_init": "k-means++",
            "stability_runs": int(getattr(args, "stability_runs", 0)),
        },
    }
    with open(out_dir / "run_config.json", "w") as f:
        json.dump(config, f, indent=2)

    method_reports = []
    for method in args.cluster_methods:
        print(f"\n=== Starting separate HPO for method: {method} ===")
        method_out = out_dir / method
        method_out.mkdir(parents=True, exist_ok=True)

        sampler = optuna.samplers.TPESampler(seed=args.seed)
        study = optuna.create_study(
            study_name=f"{args.study_name}_{method}",
            direction="maximize",
            sampler=sampler,
            storage=storage,
            load_if_exists=bool(storage),
        )
        study.optimize(
            make_objective(method, X_healthy, groups, args, device),
            n_trials=args.n_trials,
            n_jobs=args.n_jobs,
            gc_after_trial=True,
        )

        trials_df = trial_dataframe(study)
        trials_df.to_csv(method_out / "optuna_trials.csv", index=False)

        best_params = dict(study.best_trial.params)
        best_info = {
            "method": method,
            "best_value_composite_objective": study.best_value,
            "best_trial_number": study.best_trial.number,
            "best_params": best_params,
            "best_trial_user_attrs": study.best_trial.user_attrs,
        }
        with open(method_out / "best_trial.json", "w") as f:
            json.dump(best_info, f, indent=2)

        print(f"Best {method} trial:")
        print(json.dumps(best_info, indent=2))

        print(f"\n=== Final retraining for method: {method} ===")
        report = final_run_for_method(
            method=method,
            best_params=best_params,
            records_healthy=records_healthy,
            records_st=records_st,
            X_healthy=X_healthy,
            X_st=X_st,
            groups=groups,
            args=args,
            device=device,
            method_out_dir=method_out,
        )
        method_reports.append(report)

    comparison_rows = []
    for rep in method_reports:
        row = {
            "method": rep["method"],
            "final_objective_composite_healthy_reference": rep["final_objective_composite_healthy_reference"],
            "final_silhouette_healthy_reference": rep["final_silhouette_healthy_reference"],
            "final_davies_bouldin_healthy_reference": rep["final_davies_bouldin_healthy_reference"],
            **{f"best_{k}": v for k, v in rep["best_params"].items()},
        }
        comparison_rows.append(row)

    comp_df = pd.DataFrame(comparison_rows).sort_values("final_objective_composite_healthy_reference", ascending=False)
    comp_df.to_csv(out_dir / "method_comparison_summary.csv", index=False)

    best_overall = comp_df.iloc[0].to_dict() if not comp_df.empty else {}
    with open(out_dir / "best_overall_method.json", "w") as f:
        json.dump(best_overall, f, indent=2)

    print("\n=== Method comparison ===")
    print(comp_df)
    print(f"\nSaved all outputs to: {out_dir}")


## Notebook run configuration

In [ ]:
# Notebook run configuration
from types import SimpleNamespace

args = SimpleNamespace(
    data_dir=r"C:\Users\ebenc\PycharmProjects\Project AI in TimeSeries\sleep-phenotype-discovery\sleep-edf-database",
    out_dir=r"C:\Users\ebenc\PycharmProjects\Project AI in TimeSeries\sleep-phenotype-discovery",
    cluster_methods=parse_cluster_methods("kmeans,gmm,agglomerative"),
    n_trials=50,
    epochs=20,
    final_epochs=40,
    batch_size=16,
    seed=42,
    study_name="sleepedf_hpo",
    storage="sqlite:///study.db",           # Stored automatically inside hpo_results/study.db
    umap_seed=42,
    stability_runs=1,
    n_jobs=4,                               # Use more jobs for faster CPU-based HPO; keep 1 for GPU use.
    st_placebo_night_digit="1",
    st_temazepam_night_digit="2",
)

# Start with:
run_pipeline(args)


Device: cpu
Writing all outputs to: C:\Users\ebenc\PycharmProjects\Project AI in TimeSeries\sleep-phenotype-discovery\hpo_results
Loading preprocessed FIF records from: C:\Users\ebenc\PycharmProjects\Project AI in TimeSeries\sleep-phenotype-discovery\sleep-edf-database


[I 2026-05-16 19:03:20,697] Using an existing study with name 'sleepedf_hpo_kmeans' instead of creating a new one.


Loaded records: total=197, SC=153, ST=44

=== Starting separate HPO for method: kmeans ===


[I 2026-05-16 19:03:49,120] Trial 176 finished with value: 0.774788757101309 and parameters: {'latent_dim': 128, 'n_conv_blocks': 3, 'filters_base': 32, 'kernel_size': 3, 'learning_rate': 0.00011841866269497895, 'n_clusters': 3, 'umap_n_neighbors': 5, 'umap_n_components': 2, 'umap_min_dist': 0.0}. Best is trial 54 with value: 0.8951553157296668.
[I 2026-05-16 19:04:19,097] Trial 177 finished with value: -999.0 and parameters: {'latent_dim': 128, 'n_conv_blocks': 3, 'filters_base': 32, 'kernel_size': 3, 'learning_rate': 0.00012971562657354427, 'n_clusters': 6, 'umap_n_neighbors': 5, 'umap_n_components': 2, 'umap_min_dist': 0.0}. Best is trial 54 with value: 0.8951553157296668.
[I 2026-05-16 19:04:47,952] Trial 178 finished with value: 0.6631482047389393 and parameters: {'latent_dim': 128, 'n_conv_blocks': 3, 'filters_base': 32, 'kernel_size': 3, 'learning_rate': 0.0004924185908787295, 'n_clusters': 3, 'umap_n_neighbors': 5, 'umap_n_components': 2, 'umap_min_dist': 0.0}. Best is trial 54

Best kmeans trial:
{
  "method": "kmeans",
  "best_value_composite_objective": 0.8952655922000559,
  "best_trial_number": 210,
  "best_params": {
    "latent_dim": 128,
    "n_conv_blocks": 3,
    "filters_base": 32,
    "kernel_size": 3,
    "learning_rate": 0.00011583143439602522,
    "n_clusters": 5,
    "umap_n_neighbors": 5,
    "umap_n_components": 2,
    "umap_min_dist": 0.0
  },
  "best_trial_user_attrs": {
    "best_val_loss": 1.2828004360198975,
    "cluster_counts": {
      "0": 26,
      "1": 38,
      "2": 23,
      "3": 13,
      "4": 21
    },
    "davies_bouldin": 0.195795513170628,
    "epochs_run": 20,
    "filters_base": 32,
    "kernel_size": 3,
    "latent_dim": 128,
    "learning_rate": 0.00011583143439602522,
    "n_clusters": 5,
    "n_conv_blocks": 3,
    "objective_composite": 0.8952655922000559,
    "silhouette": 0.8411045074462891,
    "trial_seed": 252,
    "umap_min_dist": 0.0,
    "umap_n_components": 2,
    "umap_n_neighbors": 5,
    "umap_seed": 42
  }


...

Best gmm trial:
{
  "method": "gmm",
  "best_value_composite_objective": 0.691982512778083,
  "best_trial_number": 37,
  "best_params": {
    "latent_dim": 16,
    "n_conv_blocks": 3,
    "filters_base": 32,
    "kernel_size": 3,
    "learning_rate": 0.002849413308823382,
    "n_clusters": 3,
    "umap_n_neighbors": 5,
    "umap_n_components": 2,
    "umap_min_dist": 0.1
  },
  "best_trial_user_attrs": {
    "best_val_loss": 0.9545404016971588,
    "cluster_counts": {
      "0": 41,
      "1": 49,
      "2": 33
    },
    "davies_bouldin": 0.7243873822598507,
    "epochs_run": 20,
    "filters_base": 32,
    "kernel_size": 3,
    "latent_dim": 16,
    "learning_rate": 0.002849413308823382,
    "n_clusters": 3,
    "n_conv_blocks": 3,
    "objective_composite": 0.691982512778083,
    "silhouette": 0.48002195358276367,
    "trial_seed": 1079,
    "umap_min_dist": 0.1,
    "umap_n_components": 2,
    "umap_n_neighbors": 5,
    "umap_seed": 42
  }
}

=== Final retraining for method: gmm ==

[I 2026-05-16 19:39:20,378] A new study created in RDB with name: sleepedf_hpo_agglomerative



=== Starting separate HPO for method: agglomerative ===


[I 2026-05-16 19:39:25,289] Trial 0 finished with value: 0.6225210231216859 and parameters: {'latent_dim': 32, 'n_conv_blocks': 1, 'filters_base': 16, 'kernel_size': 7, 'learning_rate': 0.00020589728197687926, 'n_clusters': 5, 'umap_n_neighbors': 10, 'umap_n_components': 5, 'umap_min_dist': 0.1}. Best is trial 0 with value: 0.6225210231216859.
[I 2026-05-16 19:39:38,112] Trial 1 finished with value: 0.5414008625301527 and parameters: {'latent_dim': 128, 'n_conv_blocks': 1, 'filters_base': 32, 'kernel_size': 7, 'learning_rate': 0.0022038218939289885, 'n_clusters': 3, 'umap_n_neighbors': 10, 'umap_n_components': 5, 'umap_min_dist': 0.3}. Best is trial 0 with value: 0.6225210231216859.
[I 2026-05-16 19:40:05,259] Trial 2 finished with value: 0.51836068165362 and parameters: {'latent_dim': 64, 'n_conv_blocks': 2, 'filters_base': 64, 'kernel_size': 3, 'learning_rate': 0.0016015312171361203, 'n_clusters': 4, 'umap_n_neighbors': 10, 'umap_n_components': 5, 'umap_min_dist': 0.1}. Best is trial

Best agglomerative trial:
{
  "method": "agglomerative",
  "best_value_composite_objective": 0.8505276794430218,
  "best_trial_number": 48,
  "best_params": {
    "latent_dim": 16,
    "n_conv_blocks": 3,
    "filters_base": 32,
    "kernel_size": 7,
    "learning_rate": 0.00012870260280356825,
    "n_clusters": 4,
    "umap_n_neighbors": 5,
    "umap_n_components": 2,
    "umap_min_dist": 0.0
  },
  "best_trial_user_attrs": {
    "best_val_loss": 1.282131552696228,
    "cluster_counts": {
      "0": 67,
      "1": 30,
      "2": 20,
      "3": 5
    },
    "davies_bouldin": 0.27090427614591434,
    "epochs_run": 20,
    "filters_base": 32,
    "kernel_size": 7,
    "latent_dim": 16,
    "learning_rate": 0.00012870260280356825,
    "n_clusters": 4,
    "n_conv_blocks": 3,
    "objective_composite": 0.8505276794430218,
    "silhouette": 0.7556436657905579,
    "trial_seed": 2090,
    "umap_min_dist": 0.0,
    "umap_n_components": 2,
    "umap_n_neighbors": 5,
    "umap_seed": 42
  }
}

